In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge 
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit


In [2]:
df_train = pd.read_csv('../datasets/processed/train_lag.csv')
df_test = pd.read_csv('../datasets/processed/eval_lag.csv')
cumulative_contribution = pd.read_csv('../result/cumulative_contribution.csv')

In [3]:
df_train.info()
df_train.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6055 entries, 0 to 6054
Columns: 139 entries, timestamp to output_hybrid_lag_72
dtypes: float64(136), int64(2), object(1)
memory usage: 6.4+ MB


,timestamp,temperature,humidity,surface_radiation,upper_atmospheric_radiation,hour,month,air_density,wind_velocity,output_hybrid,...,output_hybrid_lag_25,output_hybrid_lag_26,output_hybrid_lag_46,output_hybrid_lag_47,output_hybrid_lag_48,output_hybrid_lag_49,output_hybrid_lag_50,output_hybrid_lag_70,output_hybrid_lag_71,output_hybrid_lag_72
0,2019-01-04 08:00:00,-14.599,0.001,72.765,214.723,8,1,1.322,5.683,1042.513,...,517.816,79.664,3828.394,3145.506,1504.637,230.074,11.803,2418.649,1940.806,1461.490
1,2019-01-04 09:00:00,-13.791,0.001,145.273,351.385,9,1,1.325,5.768,1276.272,...,2018.503,517.816,4062.046,3828.394,3145.506,1504.637,230.074,2643.376,2418.649,1940.806
2,2019-01-04 10:00:00,-12.917,0.001,198.601,436.617,10,1,1.328,6.510,1959.656,...,3288.170,2018.503,4105.654,4062.046,3828.394,3145.506,1504.637,2466.902,2643.376,2418.649
3,2019-01-04 11:00:00,-12.343,0.001,217.455,464.588,11,1,1.330,7.488,2685.257,...,3647.134,3288.170,3822.955,4105.654,4062.046,3828.394,3145.506,1861.908,2466.902,2643.376
4,2019-01-04 12:00:00,-12.295,0.001,200.135,433.386,12,1,1.333,8.004,2829.141,...,3866.962,3647.134,2941.506,3822.955,4105.654,4062.046,3828.394,1183.530,1861.908,2466.902


In [4]:
target = 'output_hybrid'
temp = [target, 'timestamp'] 

y_train = df_train[target]
y_test = df_test[target]

feature_base = [
    'temperature', 
    'humidity', 
    'surface_radiation', 
    'upper_atmospheric_radiation', 
    'air_density', 
    'wind_velocity', 
    'hour', 
    'month'
]

model_thresholds = {
    'RidgeRegression': 1.00,  
    'RandomForest': 0.85,      
    'XGBoost': 0.95,            
    'LightGBM': 0.85,          
    'SVR': 0.95                
}

def calculate_maape(y_true, y_pred):
    epsilon = 1e-10
    return np.mean(np.arctan(np.abs((y_true - y_pred) / (np.abs(y_true) + epsilon)))) * 100

param_grids = {
    'RidgeRegression': {
        'alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
        'solver': ['svd', 'cholesky', 'lsqr', 'sag']
    }, 
    
    'RandomForest': {
        'n_estimators': [50, 100],
        'max_depth': [10, 20, None]
    },
    
    'XGBoost': {
        'n_estimators': [100, 150],
        'learning_rate': [0.05, 0.1],
        'max_depth': [4, 6]
    },
    
    'LightGBM': {
        'n_estimators': [100, 150],
        'learning_rate': [0.05, 0.1],
        'max_depth': [4, 6]
    },
    
    'SVR': {
        'kernel': ['rbf', 'linear'],
        'C': [0.1, 1.0, 10],
        'epsilon': [0.01, 0.1], 
        'gamma': ['scale', 'auto', 0.1, 0.01]
    }
}

list_model = {
    'RidgeRegression': Ridge(alpha=1.0),
    'RandomForest': RandomForestRegressor(random_state=42, n_jobs=5),
    'XGBoost': XGBRegressor(random_state=42, n_jobs=5),
    'LightGBM': LGBMRegressor(random_state=42, n_jobs=5, verbose=-1), 
    'SVR': SVR()
}

results_records = []

print("Starting the dynamic feature selection and hyperparameter tuning process...")

for model_name, regressor in list_model.items():
    current_threshold = model_thresholds[model_name]
    
    if current_threshold == 1.00:
        fitur_lolos = cumulative_contribution['Feature'].tolist()
    else:
        idx_cutoff = (cumulative_contribution['Cumulative_Contribution'] >= current_threshold).idxmax()
        fitur_lolos = cumulative_contribution['Feature'].iloc[:idx_cutoff + 1].tolist()
    
    fitur_lag_lolos = [col for col in fitur_lolos if 'lag' in col]
    
    fitur_gabungan = feature_base + fitur_lag_lolos
        
    fitur_aktif = [col for col in fitur_gabungan if col in df_train.columns and col not in temp]
    
    num_lag_only = len([c for c in fitur_aktif if 'lag' in c])
    
    print(f"-> Model: {model_name} | Threshold Lag: {current_threshold * 100}%")
    print(f"-> Number of features used: {len(fitur_aktif)} (Base: {len(feature_base)} fitur, Lag: {num_lag_only} fitur)\n")
    
    X_train_model = df_train[fitur_aktif]
    X_test_model = df_test[fitur_aktif]
    
    scaler_X = MinMaxScaler(feature_range=(0, 1))
    scaler_y = MinMaxScaler(feature_range=(0, 1))
    
    X_train_scaled = scaler_X.fit_transform(X_train_model)
    X_test_scaled = scaler_X.transform(X_test_model)
    y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
    
    tscv = TimeSeriesSplit(n_splits=3)
    grid_search = GridSearchCV(
        estimator=regressor,
        param_grid=param_grids[model_name],
        cv=tscv, 
        scoring='neg_root_mean_squared_error', 
        verbose=1
    )
    
    grid_search.fit(X_train_scaled, y_train_scaled)
    best_model = grid_search.best_estimator_
    print(f"   Best Parameters {model_name}: {grid_search.best_params_}")
    
    preds_scaled = best_model.predict(X_test_scaled)
    preds = scaler_y.inverse_transform(preds_scaled.reshape(-1, 1)).flatten()
    
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    maape = calculate_maape(y_test, preds)
    r2 = r2_score(y_test, preds)

    results_records.append({
        'Model': model_name,
        'Threshold (%)': f"{current_threshold * 100}%",
        'Total_Features': len(fitur_aktif),
        'Best_Params': str(grid_search.best_params_),
        'RMSE': rmse,
        'MAE': mae, 
        'MAAPE (%)': maape, 
        'R2': r2,
    })

df_result_summary = pd.DataFrame(results_records)

print("Summary of Model Performance Evaluation (Tuned via TSCV)\n")
print(df_result_summary.to_string(index=False, formatters={
    'RMSE': '{:,.4f}'.format, 
    'MAE': '{:,.4f}'.format,
    'MAAPE (%)': '{:,.2f}%'.format,
    'R2': '{:,.4f}'.format
}))


Starting the dynamic feature selection and hyperparameter tuning process...
-> Model: RidgeRegression | Threshold Lag: 100.0%
-> Number of features used: 73 (Base: 8 fitur, Lag: 65 fitur)

Fitting 3 folds for each of 24 candidates, totalling 72 fits


C:\Users\User KBJ 14\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\User KBJ 14\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\User KBJ 14\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\User KBJ 14\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_

   Parameter Terbaik RidgeRegression: {'alpha': 0.1, 'solver': 'lsqr'}
-> Model: RandomForest | Threshold Lag: 85.0%
-> Number of features used: 14 (Base: 8 fitur, Lag: 6 fitur)

Fitting 3 folds for each of 6 candidates, totalling 18 fits
   Parameter Terbaik RandomForest: {'max_depth': 20, 'n_estimators': 100}
-> Model: XGBoost | Threshold Lag: 95.0%
-> Number of features used: 35 (Base: 8 fitur, Lag: 27 fitur)

Fitting 3 folds for each of 8 candidates, totalling 24 fits
   Parameter Terbaik XGBoost: {'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 150}
-> Model: LightGBM | Threshold Lag: 85.0%
-> Number of features used: 14 (Base: 8 fitur, Lag: 6 fitur)

Fitting 3 folds for each of 8 candidates, totalling 24 fits


C:\Users\User KBJ 14\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\User KBJ 14\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\User KBJ 14\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\User KBJ 14\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packag

   Parameter Terbaik LightGBM: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 150}
-> Model: SVR | Threshold Lag: 95.0%
-> Number of features used: 35 (Base: 8 fitur, Lag: 27 fitur)

Fitting 3 folds for each of 48 candidates, totalling 144 fits
   Parameter Terbaik SVR: {'C': 10, 'epsilon': 0.01, 'gamma': 0.01, 'kernel': 'rbf'}
Summary of Model Performance Evaluation (Tuned via TSCV)

          Model Threshold (%)  Total_Features                                                 Best_Params     RMSE      MAE MAAPE (%)     R2
RidgeRegression        100.0%              73                            {'alpha': 0.1, 'solver': 'lsqr'} 304.0884 202.3898    13.44% 0.9741
   RandomForest         85.0%              14                      {'max_depth': 20, 'n_estimators': 100} 286.3632 154.5792     6.36% 0.9770
        XGBoost         95.0%              35 {'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 150} 293.5613 181.5536    10.43% 0.9758
       LightGBM         85.0%           